# Vector Autoregression, Granger Causality, GARCH & Calendar Effects

**Author:** Antonio
**Course:** MSc Financial Data Science, UCD Smurfit Business School

## Executive summary

This notebook covers four connected time-series techniques applied to real market data:

1. **VAR (Vector Autoregression)** and **impulse response analysis** across Airbus (AIR.PA), Bank
   of America (BAC), the CAC 40 index, and the VIX.
2. **Granger causality testing** across the same four series.
3. A **GARCH(1,1)** volatility model fitted to Cronos (CRO-USD) crypto returns.
4. A **day-of-week seasonality study** on CAC 40 returns, 2005-2024.

**Key findings**
- All four return/level-change series (Airbus, BofA, CAC40, VIX) are stationary (ADF p ≈ 0), so a
  VAR in returns/first differences is appropriate without further differencing.
- Lag-order selection criteria disagree (AIC/FPE prefer 16 lags, HQIC prefers 6, BIC prefers 2) —
  a common outcome in daily-frequency financial VARs, and worth flagging explicitly rather than
  picking the criterion that gives the "nicest" answer.
- Airbus is the most externally-driven series in the system: it is significantly affected by its
  own past, by Bank of America, by the CAC 40, and by the VIX.
- The Granger causality matrix shows broad-based interdependence, with p-values below 1% for most
  pairs — most strongly, CAC40 and VIX Granger-cause all three other series.
- Cronos (CRO-USD) returns show a highly significant ARCH effect and a GARCH(1,1) fit with
  α + β ≈ 0.98, i.e. volatility is highly persistent (shocks decay slowly) but the process remains
  stationary in variance.
- CAC 40 returns show a mild "Tuesday/Wednesday effect" (average returns of +0.06%/+0.07% vs.
  -0.03%/-0.02% on Monday/Thursday) but **none of these weekday deviations are statistically
  significant** (all t-test p-values > 0.2) — a useful reminder to distinguish a visually appealing
  pattern from a statistically robust one.

**Fixes applied vs. the original draft:** the final GARCH-vs-returns overlay plot referenced an
undefined variable (`gm_std`) that would crash a fresh kernel — it now correctly uses
`garch_stdev` (the model's conditional volatility). Repeated ADF-test and plotting code has been
refactored into helper functions.

**How to reproduce:** run all cells top to bottom. Requires an internet connection (Yahoo Finance
via `yfinance`) and the `arch` package for the GARCH model. Chart PNGs are saved into a local
`charts/` folder as each figure is produced.


In [1]:
import warnings
warnings.simplefilter(action='ignore', category=RuntimeWarning)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns
import calendar
from functools import partial
import operator

import statsmodels.formula.api as smf
import statsmodels.api as sm
import statsmodels.tsa.api as smt
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.stats.diagnostic import het_arch
from statsmodels.compat import lzip
from scipy import stats
from arch import arch_model

os.makedirs('charts', exist_ok=True)


In [3]:
udinese = "Forza Udinse"
print(udinese)

Forza Udinse


## Helper functions

In [ ]:
def run_adf_test(series, name):
    """Run an Augmented Dickey-Fuller test and print a short verdict."""
    result = adfuller(series)
    print(f'--- ADF test: {name} ---')
    print(f'ADF statistic: {result[0]:.6f}')
    print(f'p-value:       {result[1]:.6g}')
    verdict = 'stationary' if result[1] <= 0.05 else 'not stationary'
    print(f'Verdict (5% level): series is {verdict}\n')
    return result


def plot_series(series, label, color, title, filename=None):
    plt.figure()
    series.plot(label=label, color=color)
    plt.title(title)
    plt.ylabel('Price')
    plt.xlabel('Date')
    plt.legend()
    if filename:
        plt.savefig(f'charts/{filename}')
    plt.show()


## 1. Data: Airbus, Bank of America, CAC 40, VIX

Bank of America was chosen to see whether Warren Buffett's well-publicised position changes (he
began building a stake in 2017 and trimmed roughly $15bn / 375.76m shares of it in 2024) line up
with anything visible in the return data. Airbus is included as the world's largest aerospace
group by deliveries (761 aircraft delivered in 2024 vs. 333 for Boeing) and a top-6 constituent of
the CAC 40 by market cap, to test how much it drives the wider French index. The VIX is included as
a standard proxy for market-wide fear/uncertainty.

In [ ]:
tickers = ['BAC', 'AIR.PA', '^VIX', '^FCHI']
prices = yf.download(tickers, start='2016-1-1', end='2024-12-31')['Close']
prices = pd.DataFrame(prices)
prices.to_csv('PricesDataFrame.csv')
prices.head()


In [ ]:
plot_series(prices['AIR.PA'], 'Airbus', 'black', 'Airbus price', 'airbus_price.png')
plot_series(prices['BAC'], 'Bank of America', 'red', 'Bank of America price', 'bac_price.png')
plot_series(prices['^FCHI'], 'CAC 40', 'blue', 'CAC 40 index price', 'cac40_price.png')
plot_series(prices['^VIX'], 'VIX', 'limegreen', 'VIX index price', 'vix_price.png')


Airbus, the CAC 40 and Bank of America all trend upward over the period despite sharp
drawdowns around Covid-19 (Bank of America also had a separate negative stretch in 2022-2024). The
VIX is spiky by construction, with its largest cluster of moves around 2020 and further spikes late
in 2024.

## 2. Returns / first differences and stationarity

In [ ]:
airbus_ret = np.log(prices['AIR.PA']).diff(1).dropna()
bofa_ret = np.log(prices['BAC']).diff(1).dropna()
cac40_ret = np.log(prices['^FCHI']).diff(1).dropna()
vix_diff = prices['^VIX'].diff(1).dropna()   # VIX is a level/vol index -> first difference, not log return

for series, name in [(airbus_ret, 'Airbus'), (bofa_ret, 'Bank of America'),
                     (cac40_ret, 'CAC 40'), (vix_diff, 'VIX')]:
    run_adf_test(series, name)


All four series reject the unit-root null comfortably (VIX most strongly, at roughly -16.6;
Airbus least strongly, at roughly -8.6), confirming that returns / first differences are
appropriate inputs for the VAR without further transformation.

In [ ]:
returns = pd.DataFrame({
    'AIRBUS': airbus_ret,
    'BOfA': bofa_ret,
    'CAC40': cac40_ret,
    'VIX': vix_diff,
}).dropna()
returns.tail()


## 3. Vector Autoregression (VAR)

We start with a single-lag VAR to establish a baseline, then use formal lag-order selection before
re-fitting.

In [ ]:
var_model = smt.VAR(returns)
var_results_1lag = var_model.fit()
print(var_results_1lag.summary())


Even at a single lag, Airbus is significantly affected by all four series' own lagged values:
positively by its own and Bank of America's past returns, and negatively by the CAC 40 and the VIX.
Bank of America is only weakly (10% level) and positively affected by lagged VIX moves. The CAC 40
is hit negatively by both indices' own lagged behaviour, most strongly by itself. The VIX is
positively affected by lagged Bank of America returns (5% level) and negatively by its own lag (1%
level).

In [ ]:
order_selection = var_model.select_order(maxlags=20)
print(order_selection.summary())


The four information criteria disagree on the "optimal" lag: AIC and FPE both point to 16
lags, HQIC points to 6, and BIC points to only 2. This is a common tension in daily-frequency
financial VARs — for the impulse-response and Granger analysis below we proceed with the AIC/FPE
recommendation of 16 lags, but the disagreement itself is worth reporting rather than hiding.

In [ ]:
var_results_16lag = var_model.fit(maxlags=16)
print(var_results_16lag.summary())


With 16 lags, Airbus remains the most externally driven series — significantly (positively or
negatively) affected by its own history and by all three other series across multiple lags. The
VIX is the single strongest driver of Airbus, with several significant lags at the 1% level. The
residual correlation matrix also shows Airbus and CAC 40 residuals are the most tightly linked
(~0.72), consistent with Airbus being a large CAC 40 constituent.

In [ ]:
impulse_response = var_results_16lag.irf(16)
fig = impulse_response.plot()
fig.set_dpi(150)
plt.savefig('charts/var_impulse_response.png')
plt.show()


The impulse responses tell a consistent story: Airbus reacts strongly and positively to its
own shock at lag 1, which reverses to a negative effect almost immediately, with a second, smaller
negative shock around lag 5. Bank of America and the CAC 40 both affect Airbus negatively early on.
Airbus and the CAC 40 in turn positively affect Bank of America around lag 10. The CAC 40's
strongest response is to its own shock at lag 1 (fully dissipating by lag 3), and it is dragged down
modestly by the VIX. The VIX itself shows its largest response to a shock in Airbus around lag 10 and
to the CAC 40 mid-sample, both an order of magnitude larger than the responses seen among the other
three series — consistent with the VIX capturing broad market stress rather than single-name
effects.

## 4. Granger causality

In [ ]:
def granger_matrix(data, maxlag=15, test='ssr_chi2test'):
    """Pairwise Granger-causality matrix: cell (row=y, col=x) is the minimum p-value
    across lags 1..maxlag for the hypothesis that x Granger-causes y."""
    variables = data.columns
    result = pd.DataFrame(np.zeros((len(variables), len(variables))),
                           columns=variables, index=variables)
    for x_col in variables:
        for y_row in variables:
            test_result = grangercausalitytests(data[[y_row, x_col]], maxlag=maxlag, verbose=False)
            p_values = [round(test_result[lag + 1][0][test][1], 4) for lag in range(maxlag)]
            result.loc[y_row, x_col] = np.min(p_values)
    result.columns = [f'{v}_x' for v in variables]
    result.index = [f'{v}_y' for v in variables]
    return result

granger_matrix(returns)


P-values below 0.01 dominate the matrix, indicating broad-based Granger causality across the
system: for example Bank of America and Airbus Granger-cause each other, the CAC 40 Granger-causes
all three other series (p ≈ 0), and the VIX Granger-causes all three other series as well. VIX's
effect on Airbus and Bank of America is significant at the 5% (not 1%) level, a somewhat weaker
result than the other relationships in the matrix.

## 5. GARCH volatility modelling: Cronos (CRO-USD)

Cronos was chosen for its role as the native token of the Crypto.com exchange and its focus on
low-cost, high-throughput cross-chain transactions.

In [ ]:
crypto_price = yf.download('CRO-USD', start='2019-01-01', end='2024-12-31')['Close']
crypto_returns = np.log(crypto_price).diff(1).dropna()
crypto_returns = pd.DataFrame(crypto_returns)
crypto_returns.to_csv('CronosReturns.csv')

crypto_returns.plot(figsize=(10, 6), color='navy', legend=False, title='Cronos (CRO-USD) daily log returns')
plt.xlabel('Date')
plt.ylabel('Returns')
plt.savefig('charts/cronos_returns.png')
plt.show()


Returns were strongly positive through 2019 (especially late Q1), turned negative around the
Covid-19 shock in early 2020, rallied hard through 2021, then gave much of it back over 2022-2023,
before another sharp positive move in late 2024 coinciding with a Crypto.com / Google Cloud
partnership announcement.

In [ ]:
mean_model = sm.OLS(crypto_returns, sm.add_constant(crypto_returns)['const']).fit()
print(mean_model.summary())


The constant-only mean model is not statistically significant (p = 0.481): there is no
evidence the average daily return differs from zero. The residuals show strong positive skew
(2.16) and very high excess kurtosis (38.9, vs. 3 for a normal distribution) — a right-skewed, very
fat-tailed distribution typical of crypto assets.

In [ ]:
residuals = mean_model.resid

plt.figure(figsize=(10, 5))
plt.hist(residuals, bins=150, color='navy', label='Cronos returns (mean-adjusted)')
plt.legend()
plt.savefig('charts/cronos_hist.png')
plt.show()


In [ ]:
arch_test = het_arch(residuals, nlags=5)
lzip(['lm', 'lm p-val', 'f val', 'f p-val'], arch_test)


The Lagrange-multiplier ARCH test rejects the null of no ARCH effects overwhelmingly
(p ≈ 2.6e-32), confirming the returns exhibit volatility clustering that a constant-variance model
would miss.

In [ ]:
garch_model = arch_model(crypto_returns, p=1, q=1, vol='GARCH', dist='normal')
garch_results = garch_model.fit(disp='off')
print(garch_results.summary())


The mean (mu) is statistically indistinguishable from zero (p = 0.809). In the volatility
equation, both alpha (shock impact, 0.20) and beta (volatility persistence, 0.78) are highly
significant, and alpha + beta ≈ 0.98 — just below 1, indicating a stationary but highly persistent
(slow-decaying) volatility process, consistent with strong clustering.

In [ ]:
garch_residuals = garch_results.resid
garch_stdev = garch_results.conditional_volatility
standardized_residuals = garch_residuals / garch_stdev

plt.figure(figsize=(10, 4))
plt.hist(standardized_residuals, bins=50, facecolor='navy', label='Standardized residuals')
plt.legend()
plt.savefig('charts/garch_standardized_resid_hist.png')
plt.show()

plt.figure(figsize=(10, 4))
plt.hist(garch_residuals, bins=50, facecolor='royalblue', label='Raw (non-standardized) residuals')
plt.legend()
plt.savefig('charts/garch_raw_resid_hist.png')
plt.show()


If the GARCH model fit perfectly, the standardized residuals should look close to a standard
normal. In practice they remain somewhat left-skewed with fat tails, meaning the model captures
most — but not all — of the time-varying volatility; some extreme moves remain unexplained. The raw
(non-standardized) residuals are far more tightly clustered near zero, showing that the GARCH
model has already absorbed most of the variance structure before standardization.

In [ ]:
garch_results.plot()
plt.savefig('charts/garch_std_resid_and_cond_vol.png')
plt.show()


The standardized-residual series fluctuates around zero with the largest spikes around and
during 2020, plus one notable spike at the end of 2024 — periods where the model's volatility
forecast was furthest from realised moves. The conditional-volatility series shows clear clustering,
with the most prominent spike around the broad 2020 market shock.

In [ ]:
# NOTE: the original draft referenced an undefined variable `gm_std` here, which would raise
# a NameError on a fresh run. The correct series is `garch_stdev` (conditional_volatility), fixed below.
plt.figure(figsize=(12, 6))
plt.plot(crypto_returns, color='royalblue', alpha=0.4, label='CRO-USD returns')
plt.plot(garch_stdev, color='navy', label='GARCH conditional volatility')
plt.legend()
plt.savefig('charts/cronos_returns_vs_garch_vol.png')
plt.show()


The GARCH conditional-volatility line tracks the envelope of the raw returns well, rising
during clusters of large moves (e.g. mid-2019 and late 2021) and staying low during the calmer
stretches in between — exactly the clustering behaviour the model is designed to capture.

## 6. CAC 40 day-of-week effect (2005-2024)

A classic "calendar anomaly" test: are average returns systematically different by weekday?

In [ ]:
cac40_price = yf.download('^FCHI', start='2005-01-01', end='2024-12-31')['Close']

plt.figure(figsize=(14, 6))
plt.plot(cac40_price, color='forestgreen')
plt.ylabel('CAC 40 price')
plt.xlabel('Date')
plt.title('CAC 40 index, 2005-2024')
plt.savefig('charts/cac40_long_history.png')
plt.show()


The index gained roughly 85% over the two decades shown, despite three major drawdowns: the
2008 financial crisis (-58% peak to trough), the 2011 European debt-crisis selloff (roughly -20% in
ten trading days), and the 2020 Covid-19 shock. The 2022 energy crisis linked to the war in Ukraine
weighed on the index through 2023 before a recovery into 2024.

In [ ]:
cac40_returns = cac40_price.pct_change().dropna()
cac40_returns = pd.DataFrame(cac40_returns)
cac40_returns.to_csv('CAC40_returns.csv')

cac40_returns['Weekday'] = cac40_returns.index.weekday.map(lambda x: calendar.day_name[x])
cac40_returns.to_csv('CAC40_returns_weekday.csv')
cac40_returns.tail()


In [ ]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
weekday_mean = cac40_returns.groupby('Weekday')['^FCHI'].mean().reindex(weekday_order)
weekday_mean


On average, Monday and Thursday are (very slightly) negative days for the CAC 40, while
Tuesday, Wednesday, and Friday are (very slightly) positive — Wednesday is the best day and Monday
the worst.

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=weekday_mean.index, y=100 * weekday_mean.values, palette='pastel')
plt.ylabel('Average return (%)')
plt.title('Average daily CAC 40 return by weekday, 2005-2024')
plt.savefig('charts/cac40_weekday_bar.png')
plt.show()


In [ ]:
weekday_num = cac40_returns['Weekday'].map({d: i + 1 for i, d in enumerate(weekday_order)})

plt.figure(figsize=(12, 8))
plt.scatter(weekday_num, 100 * cac40_returns['^FCHI'], color='green', marker='x', linewidths=1, s=30)
plt.xlabel('Weekday (1 = Monday, ..., 5 = Friday)')
plt.ylabel('Return (%)')
plt.title('Daily CAC 40 returns by weekday, 2005-2024')
plt.savefig('charts/cac40_weekday_scatter.png')
plt.show()


Monday shows both the widest spread and the most extreme outliers (largest single-day
losses), consistent with it also having the highest standard deviation of returns below.

In [ ]:
weekday_std = cac40_returns.groupby('Weekday')['^FCHI'].std().reindex(weekday_order)
weekday_count = cac40_returns.groupby('Weekday')['^FCHI'].count().reindex(weekday_order)
print(weekday_std)
print(weekday_count)


Monday has the highest standard deviation of the five weekdays and also the lowest average
return — the riskiest day is also (on average) the worst-performing one. Tuesday and Wednesday
combine the smallest spread with a positive skew, consistent with their positive average returns
above.

In [ ]:
average_return = weekday_mean.mean()
ttest = partial(stats.ttest_1samp, popmean=average_return)
ttest_results = cac40_returns.groupby('Weekday')['^FCHI'].apply(ttest)

ttest_df = pd.DataFrame.from_records(
    map(operator.methodcaller('_asdict'), ttest_results), index=ttest_results.index
)
ttest_df['df'] = weekday_count - 1
ttest_df[['statistic', 'pvalue', 'df']]


None of the five weekday t-tests reject the null that a given day's mean return equals the
overall mean return across all days (all p-values > 0.2). In other words, while Monday/Thursday
look weaker and Tuesday/Wednesday look stronger in the raw averages, the effect is **not**
statistically distinguishable from noise over this 20-year sample.

In [ ]:
cac40_returns['Year'] = cac40_returns.index.year
yearly_weekday_mean = cac40_returns.groupby(['Year', 'Weekday'], as_index=False)['^FCHI'].mean()
yearly_weekday_mean['Weekday'] = pd.Categorical(yearly_weekday_mean['Weekday'],
                                                 categories=weekday_order, ordered=True)

palette = {'Monday': 'red', 'Tuesday': 'blue', 'Wednesday': 'green', 'Thursday': 'orange', 'Friday': 'purple'}

g = sns.catplot(data=yearly_weekday_mean, x='Weekday', y='^FCHI', hue='Weekday',
                 kind='bar', palette=palette, col='Year', col_wrap=4, height=3, aspect=1.33)
g.fig.suptitle('Average CAC 40 return by weekday, per year (2005-2024)', fontsize=18)
g.tight_layout()
g.fig.subplots_adjust(top=0.93)
plt.savefig('charts/cac40_weekday_by_year.png')
plt.show()


Monday is negative in most years, with the standout exceptions of 2009, 2010 and 2020, each of
which also happened to be a year of large post-crisis or crisis-driven rebounds. Thursday shows the
most year-to-year volatility, with notably strong positive years in 2015 and 2021 and notably weak
years around major shocks (2008 Great Recession, 2020 Covid-19, 2022 energy crisis). Tuesday and
Wednesday are positive in most years, with 2008 and 2024 as the main exceptions on Wednesday.
Friday is positive in nearly every year shown, with 2010 the lone exception.